# Chapter 2: Recursive State Estimation
## Discrete Bayes Filter — Pluto in the Hallway

Recreates the classic example from *Probabilistic Robotics* (Thrun et al.), pp. 27-28.

**World**: 1D hallway, 10m long, with 3 identical doors at x=2.0, 5.0, 8.0m

**Belief**: Discrete grid (100 cells, Δx=0.1m)

**Insight**: Watch how 3 peaks → 1 peak as Pluto moves + senses.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

from pluto_filters.pluto_filters.bayes_filter.discrete_bayes_filter import (
    DiscreteBayesFilter1D, CELL_CENTERS, DOOR_POSITIONS
)

In [ ]:
# ============================================================
# Scenario: Pluto moves right, senses doors 3 times
# ============================================================
bf = DiscreteBayesFilter1D()

history = []
events = []

def record(event):
    history.append(bf.belief.copy())
    events.append(event)

record('prior (uniform)')

# Step 1: sense door (at x=2, robot is actually near door 1)
bf.update(True)  
record('sense door → 3 peaks')

# Step 2: move right 3m
bf.predict(3.0)
record('move +3m → peaks shift')

# Step 3: sense door again (now near door 2)
bf.update(True)
record('sense door → 2 peaks')

# Step 4: move right 3m
bf.predict(3.0)
record('move +3m')

# Step 5: sense door (now near door 3)
bf.update(True)
record('sense door → 1 peak (localized!)')

print(f'Final mode: x = {CELL_CENTERS[np.argmax(bf.belief)]:.2f}m')
print(f'Entropy: {bf.uncertainty():.4f}')

In [ ]:
fig, axes = plt.subplots(len(history), 1, figsize=(12, 2.5 * len(history)))
fig.suptitle("Discrete Bayes Filter — Pluto's Belief Over Time", fontsize=14)

for i, (belief, event) in enumerate(zip(history, events)):
    ax = axes[i]
    ax.bar(CELL_CENTERS, belief, width=0.09, color='steelblue', alpha=0.8)
    for d in DOOR_POSITIONS:
        ax.axvline(d, color='brown', linestyle='--', alpha=0.5, label='door' if i == 0 else '')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, belief.max() * 1.3 + 1e-5)
    ax.set_ylabel('p(x)')
    ax.set_title(f'Step {i}: {event}  [entropy={-np.sum((belief+1e-300)*np.log(belief+1e-300)):.3f}]')
    if i == 0:
        ax.legend()
    if i == len(history) - 1:
        ax.set_xlabel('Position [m]')

plt.tight_layout()
plt.savefig('ch02_bayes_filter.png', dpi=120)
plt.show()
print('Saved to ch02_bayes_filter.png')

## Exercises
1. What happens if `P_HIT_DOOR = 0.5` (sensor is no better than random)?
2. Increase `MOTION_SIGMA` to 0.5. How many more steps to localize?
3. What happens if the robot moves past the last door — can it still localize?